# 05 - SIMCA Validation Robustness Review

This notebook reviews the validation-stage candidates produced by notebook 04C. It does not use the pure test split and it does not perform final model selection. Its role is to identify robustness risks and diagnostic evidence before the pure-test and final multi-model selection notebooks.


## Inputs and outputs

Inputs:

- `results/04C_simca_concat_refit_<RESULTS_TAG>/candidate_panel.parquet`
- `results/04C_simca_concat_refit_<RESULTS_TAG>/validation_refit_metrics_long.parquet`
- `results/04C_simca_concat_refit_<RESULTS_TAG>/validation_refit_2way_object_metrics.parquet`
- `results/04C_simca_concat_refit_<RESULTS_TAG>/validation_refit_2way_pixel_metrics.parquet`
- `results/04C_simca_concat_refit_<RESULTS_TAG>/validation_refit_3way_object_metrics.parquet`
- optional duplicate-diagnostic tables from notebook 04C
- optional 04C batch pixel tables for border/core diagnostics

Outputs:

- `robustness_scored_metrics.parquet`
- `robustness_primary_metrics.parquet`
- `pareto_2way_front.parquet`
- `pareto_2way_annotated.parquet`
- `pareto_2way_audit.parquet`
- `pareto_3way_front.parquet`
- `pareto_3way_annotated.parquet`
- `pareto_3way_audit.parquet`
- `ablation_diagnostics.parquet`
- `random_state_stability_panel.parquet`
- `random_state_stability_metrics.parquet`
- `random_state_stability_summary.parquet`
- `random_state_stability_errors.parquet`
- `border_core_diagnostics.parquet`
- `border_core_status.parquet`
- `duplicated_candidate_review.parquet`
- `duplicated_candidate_summary.parquet`
- `track_scoring_flags.parquet`
- `robustness_protocol.parquet`


In [2]:
from pathlib import Path
import sys
import gc
import json

CURRENT_DIR = Path.cwd().resolve()
if (CURRENT_DIR / "src").exists():
    PROJECT_ROOT = CURRENT_DIR
elif (CURRENT_DIR.parent / "src").exists():
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    raise RuntimeError(
        "Could not find project root. Run this notebook from the project root or notebooks/."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)

import numpy as np
import pandas as pd
from IPython.display import display

from src import experiment_config as expcfg
from src.io.database_h5 import load_nir_uco_h5
from src.spectra.band_selection import select_wavelength_range_from_database, wavelength_selection_summary
from src.workflows.simca import make_target_train_filters, run_selected_simca_random_state_stability
from src.workflows.simca_candidates import (
    build_pca_preprocessing_configs_by_matrix_family,
    normalize_simca_candidate_columns,
    validate_simca_candidate_contract,
)
from src.workflows.simca_tables import compact_simca_table_for_path
from src.workflows.simca_robustness import (
    add_simca_robustness_scores,
    build_ablation_diagnostics,
    build_border_core_diagnostics,
    build_border_core_skip_table,
    build_duplicated_candidate_review,
    build_pareto_diagnostics,
    build_random_state_stability_panel,
    build_track_scoring_table,
    select_track_primary_or_available_metrics,
    summarize_duplicated_candidate_review,
    summarize_random_state_stability_metrics,
    validate_simca_robustness_inputs,
)


PROJECT_ROOT: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts


## Configuration

Global switches control the expensive diagnostic blocks. Random-state stability refits candidates over several seeds and is disabled by default. Border/core diagnostics only run when notebook 04C saved detailed batch pixel tables.


In [3]:
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RESULTS_TAG = expcfg.DEFAULT_RESULTS_TAG
WAVELENGTH_MODE = expcfg.DEFAULT_WAVELENGTH_MODE

RESULTS_03_DIR = PROJECT_ROOT / "results" / f"03_pca_{RESULTS_TAG}"
RESULTS_04C_DIR = PROJECT_ROOT / "results" / f"04C_simca_concat_refit_{RESULTS_TAG}"
RESULTS_DIR = PROJECT_ROOT / "results" / f"05_simca_validation_robustness_{RESULTS_TAG}"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DB_H5_PATH = PROJECT_ROOT / "HSI Data" / "processed" / "nir_uco_database.h5"
PCA_SELECTED_PREPROCESSINGS_PATH = RESULTS_03_DIR / "pca_selected_preprocessings.parquet"

CANDIDATE_PANEL_PATH = RESULTS_04C_DIR / "candidate_panel.parquet"
VALIDATION_METRICS_LONG_PATH = RESULTS_04C_DIR / "validation_refit_metrics_long.parquet"
VALIDATION_2WAY_OBJECT_METRICS_PATH = RESULTS_04C_DIR / "validation_refit_2way_object_metrics.parquet"
VALIDATION_2WAY_PIXEL_METRICS_PATH = RESULTS_04C_DIR / "validation_refit_2way_pixel_metrics.parquet"
VALIDATION_3WAY_OBJECT_METRICS_PATH = RESULTS_04C_DIR / "validation_refit_3way_object_metrics.parquet"
METRIC_EQUIVALENCE_GROUPS_PATH = RESULTS_04C_DIR / "metric_equivalent_config_groups.parquet"
METRIC_EQUIVALENCE_DROPPED_PATH = RESULTS_04C_DIR / "metric_equivalent_config_dropped.parquet"
DUPLICATED_REFIT_METRIC_COMPARISON_PATH = RESULTS_04C_DIR / "duplicated_refit_metric_comparison.parquet"

VALIDATION_BATCH_PIXELS_DIR = RESULTS_04C_DIR / "validation_refit_batches" / "pixels"

ROBUSTNESS_SCORED_METRICS_PATH = RESULTS_DIR / "robustness_scored_metrics.parquet"
ROBUSTNESS_PRIMARY_METRICS_PATH = RESULTS_DIR / "robustness_primary_metrics.parquet"
PARETO_2WAY_FRONT_PATH = RESULTS_DIR / "pareto_2way_front.parquet"
PARETO_2WAY_ANNOTATED_PATH = RESULTS_DIR / "pareto_2way_annotated.parquet"
PARETO_2WAY_AUDIT_PATH = RESULTS_DIR / "pareto_2way_audit.parquet"
PARETO_3WAY_FRONT_PATH = RESULTS_DIR / "pareto_3way_front.parquet"
PARETO_3WAY_ANNOTATED_PATH = RESULTS_DIR / "pareto_3way_annotated.parquet"
PARETO_3WAY_AUDIT_PATH = RESULTS_DIR / "pareto_3way_audit.parquet"
ABLATION_DIAGNOSTICS_PATH = RESULTS_DIR / "ablation_diagnostics.parquet"
RANDOM_STATE_STABILITY_PANEL_PATH = RESULTS_DIR / "random_state_stability_panel.parquet"
RANDOM_STATE_STABILITY_METRICS_PATH = RESULTS_DIR / "random_state_stability_metrics.parquet"
RANDOM_STATE_STABILITY_SUMMARY_PATH = RESULTS_DIR / "random_state_stability_summary.parquet"
RANDOM_STATE_STABILITY_ERRORS_PATH = RESULTS_DIR / "random_state_stability_errors.parquet"
BORDER_CORE_DIAGNOSTICS_PATH = RESULTS_DIR / "border_core_diagnostics.parquet"
BORDER_CORE_STATUS_PATH = RESULTS_DIR / "border_core_status.parquet"
DUPLICATED_CANDIDATE_REVIEW_PATH = RESULTS_DIR / "duplicated_candidate_review.parquet"
DUPLICATED_CANDIDATE_SUMMARY_PATH = RESULTS_DIR / "duplicated_candidate_summary.parquet"
TRACK_SCORING_FLAGS_PATH = RESULTS_DIR / "track_scoring_flags.parquet"
ROBUSTNESS_PROTOCOL_PATH = RESULTS_DIR / "robustness_protocol.parquet"

TARGET_CLASS = expcfg.TARGET_CLASS
NON_TARGET_LABEL = expcfg.NON_TARGET_LABEL
REFERENCE_CLASSES = list(expcfg.REFERENCE_CLASSES)

TRAIN_FILTERS = make_target_train_filters(
    target_class=TARGET_CLASS,
    train_batches=expcfg.SIMCA_TRAIN_BATCHES,
)
VALIDATION_FILTERS = {
    "sample_kind": ["pure"],
    "object_nut_type": REFERENCE_CLASSES,
    "batch": list(expcfg.SIMCA_VALIDATION_BATCHES),
}

USE_WAVELENGTH_WINDOW = False
WINDOW_MIN_NM = 1225.0
WINDOW_MAX_NM = 1675.0

RUN_RANDOM_STATE_STABILITY = True
STABILITY_RANDOM_STATES = tuple(expcfg.SIMCA_ROBUSTNESS_RANDOM_STATES)
MAX_STABILITY_CANDIDATES_PER_TRACK = expcfg.SIMCA_ROBUSTNESS_MAX_STABILITY_CANDIDATES_PER_TRACK

RUN_BORDER_CORE_DIAGNOSTICS = True
MAX_BORDER_CORE_CONFIGS = 20

RUN_DUPLICATED_CANDIDATE_ANALYSIS = False

RANDOM_STATE = expcfg.RANDOM_STATE
REPLACE_BALANCED_PIXELS = expcfg.REPLACE_BALANCED_PIXELS
CV_N_SPLITS = expcfg.CV_N_SPLITS
CV_GROUP_COL = expcfg.CV_GROUP_COL

print("Input 04C dir:", RESULTS_04C_DIR)
print("Output dir:", RESULTS_DIR)
print("RUN_RANDOM_STATE_STABILITY:", RUN_RANDOM_STATE_STABILITY)
print("RUN_BORDER_CORE_DIAGNOSTICS:", RUN_BORDER_CORE_DIAGNOSTICS)


Input 04C dir: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04C_simca_concat_refit_non_noisy_all
Output dir: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\05_simca_validation_robustness_non_noisy_all
RUN_RANDOM_STATE_STABILITY: True
RUN_BORDER_CORE_DIAGNOSTICS: True


## Helpers

These helpers keep I/O stable and make optional inputs explicit.


In [4]:
def read_simca_parquet(path, *, required=False):
    path = Path(path)
    if not path.exists():
        if required:
            raise FileNotFoundError(path)
        return pd.DataFrame()
    return compact_simca_table_for_path(pd.read_parquet(path), path)


def read_parquet_if_exists(path):
    return read_simca_parquet(path, required=False)


def save_parquet(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    out = compact_simca_table_for_path(pd.DataFrame() if df is None else df, path)
    out.to_parquet(path, index=False)
    return path


def concat_nonempty(parts):
    parts = [part for part in parts if part is not None and len(part) > 0]
    return pd.concat(parts, ignore_index=True, sort=False) if parts else pd.DataFrame()


def load_database_for_optional_refits():
    object_db, image_db = load_nir_uco_h5(
        DB_H5_PATH,
        reconstruct_heavy_object_arrays=True,
    )
    if USE_WAVELENGTH_WINDOW:
        object_db, image_db, wavelengths, wavelength_info = select_wavelength_range_from_database(
            object_db=object_db,
            image_db=image_db,
            min_wavelength=WINDOW_MIN_NM,
            max_wavelength=WINDOW_MAX_NM,
        )
        wavelength_selection_df = wavelength_selection_summary(wavelength_info)
    else:
        first_obj = next(iter(object_db.values()))
        wavelengths = first_obj.get("wavelengths")
        wavelengths = np.asarray(wavelengths, dtype=float) if wavelengths is not None else None
        wavelength_selection_df = pd.DataFrame()

    if wavelengths is None:
        raise RuntimeError("No wavelength axis found in object_db.")
    return object_db, image_db, wavelengths, wavelength_selection_df


## Load 04C validation inputs

Only validation-stage outputs are accepted here. Any pure-test table should fail validation.


In [5]:
required_paths = [
    CANDIDATE_PANEL_PATH,
    VALIDATION_METRICS_LONG_PATH,
    VALIDATION_2WAY_OBJECT_METRICS_PATH,
    VALIDATION_2WAY_PIXEL_METRICS_PATH,
    VALIDATION_3WAY_OBJECT_METRICS_PATH,
]
missing = [str(path) for path in required_paths if not Path(path).exists()]
if missing:
    raise FileNotFoundError("Missing required notebook 04C outputs:\n" + "\n".join(missing))

candidate_panel_df = read_simca_parquet(CANDIDATE_PANEL_PATH, required=True)
candidate_panel_df = normalize_simca_candidate_columns(candidate_panel_df)
candidate_panel_df = compact_simca_table_for_path(candidate_panel_df, CANDIDATE_PANEL_PATH)

validation_metrics_long_df = read_simca_parquet(VALIDATION_METRICS_LONG_PATH, required=True)
validation_2way_object_metrics_df = read_simca_parquet(VALIDATION_2WAY_OBJECT_METRICS_PATH, required=True)
validation_2way_pixel_metrics_df = read_simca_parquet(VALIDATION_2WAY_PIXEL_METRICS_PATH, required=True)
validation_3way_object_metrics_df = read_simca_parquet(VALIDATION_3WAY_OBJECT_METRICS_PATH, required=True)

metric_equivalence_groups_df = read_parquet_if_exists(METRIC_EQUIVALENCE_GROUPS_PATH)
metric_equivalence_dropped_df = read_parquet_if_exists(METRIC_EQUIVALENCE_DROPPED_PATH)
duplicated_refit_metric_comparison_df = read_parquet_if_exists(DUPLICATED_REFIT_METRIC_COMPARISON_PATH)

validate_simca_candidate_contract(candidate_panel_df)
validation_metrics_df = validate_simca_robustness_inputs(validation_metrics_long_df)

print("candidate_panel:", candidate_panel_df.shape)
print("validation_metrics:", validation_metrics_df.shape)
print("2-way object metrics:", validation_2way_object_metrics_df.shape)
print("2-way pixel metrics:", validation_2way_pixel_metrics_df.shape)
print("3-way object metrics:", validation_3way_object_metrics_df.shape)
print("duplicate groups:", metric_equivalence_groups_df.shape)


candidate_panel: (1982, 64)
validation_metrics: (5946, 85)
2-way object metrics: (1982, 59)
2-way pixel metrics: (1982, 39)
3-way object metrics: (1982, 46)
duplicate groups: (1592, 16)


## Score candidates and build Pareto fronts

The score and flags are review diagnostics only. They rank candidates within each track for inspection; they are not final selection decisions.


In [6]:
robustness_scored_metrics_df = add_simca_robustness_scores(validation_metrics_df)
robustness_primary_metrics_df = select_track_primary_or_available_metrics(robustness_scored_metrics_df)

pareto_2way_front_df, pareto_2way_annotated_df, pareto_2way_audit_df = build_pareto_diagnostics(
    robustness_primary_metrics_df,
    decision_mode="2way",
)
pareto_3way_front_df, pareto_3way_annotated_df, pareto_3way_audit_df = build_pareto_diagnostics(
    robustness_primary_metrics_df,
    decision_mode="3way",
)

print("scored metrics:", robustness_scored_metrics_df.shape)
print("primary metrics:", robustness_primary_metrics_df.shape)
print("2-way Pareto front:", pareto_2way_front_df.shape)
print("3-way Pareto front:", pareto_3way_front_df.shape)
display(pareto_2way_audit_df)
display(pareto_3way_audit_df)


scored metrics: (5946, 89)
primary metrics: (3964, 89)
2-way Pareto front: (61, 54)
3-way Pareto front: (359, 54)


,selection_track,matrix_family,decision_mode,pareto_minimize_columns,pareto_maximize_columns,n_before,n_pareto,n_dominated
0,object_matrix_2way,object_matrix,2way,"fn_rate,fp_rate",balanced_accuracy,883,25,858
1,pixel_matrix_2way,pixel_matrix,2way,"fn_rate,fp_rate",balanced_accuracy,1099,36,1063


,selection_track,matrix_family,decision_mode,pareto_minimize_columns,pareto_maximize_columns,n_before,n_pareto,n_dominated
0,object_matrix_3way,object_matrix,3way,"target_miss_rate,non_target_false_accept_rate,...","coverage_rate,screening_sensitivity,decided_ba...",883,281,602
1,pixel_matrix_3way,pixel_matrix,3way,"target_miss_rate,non_target_false_accept_rate,...","coverage_rate,screening_sensitivity,decided_ba...",1099,78,1021


## Hyperparameter ablation diagnostics

This is an observational ablation summary over the candidates generated upstream. It does not refit altered configurations and should be interpreted as a sensitivity screen.


In [7]:
ablation_diagnostics_df = build_ablation_diagnostics(robustness_primary_metrics_df)
print("ablation diagnostics:", ablation_diagnostics_df.shape)
display(ablation_diagnostics_df.head(20))


ablation diagnostics: (131, 68)


,selection_track,matrix_family,decision_mode,metric_level,factor,factor_value,factor_value_numeric,n_configs,fn_rate_mean,fn_rate_median,...,decided_balanced_accuracy_mean,decided_balanced_accuracy_median,decided_balanced_accuracy_std,decided_balanced_accuracy_min,decided_balanced_accuracy_max,robustness_score_mean,robustness_score_median,robustness_score_std,robustness_score_min,robustness_score_max
0,object_matrix_3way,object_matrix,3way,object,balanced_pixel_strategy_effective,<NA>,NaN,883,0.052843,0.000000,...,0.697382,0.666667,0.184883,0.5,1.0,0.939600,1.355410,2.072416,-8.000000,3.497685
1,object_matrix_2way,object_matrix,2way,object,balanced_pixel_strategy_effective,random,NaN,883,0.636659,0.698113,...,NaN,NaN,NaN,NaN,NaN,-5.891542,-6.104974,2.646914,-9.163636,-0.859005
2,object_matrix_3way,object_matrix,3way,object,limit_source,<NA>,NaN,883,0.052843,0.000000,...,0.697382,0.666667,0.184883,0.5,1.0,0.939600,1.355410,2.072416,-8.000000,3.497685
3,object_matrix_2way,object_matrix,2way,object,limit_source,empirical_cv,NaN,569,0.555758,0.566038,...,NaN,NaN,NaN,NaN,NaN,-5.237034,-5.088508,2.615628,-9.109091,-0.859005
4,object_matrix_2way,object_matrix,2way,object,limit_source,chi2,NaN,227,0.756961,0.924528,...,NaN,NaN,NaN,NaN,NaN,-6.904707,-8.377358,2.465386,-9.109091,-1.961921
5,object_matrix_2way,object_matrix,2way,object,limit_source,scaled_chi2,NaN,87,0.851876,0.886792,...,NaN,NaN,NaN,NaN,NaN,-7.528630,-8.071356,1.531090,-9.163636,-3.319039
6,object_matrix_3way,object_matrix,3way,object,matrix_method,object_median,NaN,686,0.014715,0.000000,...,0.693612,0.681818,0.174071,0.5,1.0,1.324712,1.426684,1.157204,-8.000000,3.157407
7,object_matrix_3way,object_matrix,3way,object,matrix_method,object_mean,NaN,197,0.185614,0.000000,...,0.712915,0.631579,0.223322,0.5,1.0,-0.401450,0.314815,3.503252,-8.000000,3.497685
8,object_matrix_2way,object_matrix,2way,object,matrix_method,object_median,NaN,686,0.566010,0.584906,...,NaN,NaN,NaN,NaN,NaN,-5.302312,-5.132419,2.579270,-9.163636,-0.859005
9,object_matrix_2way,object_matrix,2way,object,matrix_method,object_mean,NaN,197,0.882674,0.981132,...,NaN,NaN,NaN,NaN,NaN,-7.943379,-8.792454,1.678732,-9.109091,-2.061750


## Optional random-state stability

This block refits a compact review panel over several random states. It is most useful for `balanced_pixels` candidates because their training matrix depends on sampling. Keep it disabled for quick review runs.


In [8]:
random_state_stability_panel_df = build_random_state_stability_panel(
    candidate_panel_df=candidate_panel_df,
    scored_metrics_df=robustness_primary_metrics_df,
    max_per_track=MAX_STABILITY_CANDIDATES_PER_TRACK,
)

random_state_stability_metrics_df = pd.DataFrame()
random_state_stability_summary_df = pd.DataFrame()
random_state_stability_errors_df = pd.DataFrame()

if RUN_RANDOM_STATE_STABILITY and len(random_state_stability_panel_df) > 0:
    pca_selected_preprocessings_df = pd.read_parquet(PCA_SELECTED_PREPROCESSINGS_PATH)
    preprocessing_configs_by_family = build_pca_preprocessing_configs_by_matrix_family(
        pca_selected_preprocessings_df
    )
    object_db, image_db, wavelengths, wavelength_selection_df = load_database_for_optional_refits()

    (
        random_state_stability_metrics_df,
        random_state_pixel_errors_by_image_df,
        random_state_stability_errors_df,
    ) = run_selected_simca_random_state_stability(
        selected_configs_df=random_state_stability_panel_df,
        object_db=object_db,
        image_db=image_db,
        train_filters=TRAIN_FILTERS,
        projection_filters=VALIDATION_FILTERS,
        preprocessing_configs=preprocessing_configs_by_family,
        seeds=STABILITY_RANDOM_STATES,
        evaluation_split="validation_random_state_stability",
        wavelengths=wavelengths,
        replace=REPLACE_BALANCED_PIXELS,
        cv_n_splits=CV_N_SPLITS,
        cv_group_col=CV_GROUP_COL,
        target_class=TARGET_CLASS,
        non_target_label=NON_TARGET_LABEL,
    )
    random_state_stability_summary_df = summarize_random_state_stability_metrics(
        random_state_stability_metrics_df
    )

    del object_db, image_db
    gc.collect()
else:
    print("Skipped random-state stability. Set RUN_RANDOM_STATE_STABILITY=True to run it.")

print("stability panel:", random_state_stability_panel_df.shape)
print("stability metrics:", random_state_stability_metrics_df.shape)
print("stability summary:", random_state_stability_summary_df.shape)
print("stability errors:", random_state_stability_errors_df.shape)
display(random_state_stability_panel_df.head(20))


[random_state_stability] seed=0
[validation_random_state_stability] 04C_refit_000008
[validation_random_state_stability] 04C_refit_000009
[validation_random_state_stability] 04C_refit_000002
[validation_random_state_stability] 04C_refit_000003
[validation_random_state_stability] 04C_refit_000000
[validation_random_state_stability] 04C_refit_000001
[validation_random_state_stability] 04C_refit_000004
[validation_random_state_stability] 04C_refit_000010
[validation_random_state_stability] 04C_refit_000005
[validation_random_state_stability] 04C_refit_000006
[validation_random_state_stability] 04C_refit_000036
[validation_random_state_stability] 04C_refit_000037
[validation_random_state_stability] 04C_refit_000520
[validation_random_state_stability] 04C_refit_000692
[validation_random_state_stability] 04C_refit_000728
[validation_random_state_stability] 04C_refit_000758
[validation_random_state_stability] 04C_refit_000690
[validation_random_state_stability] 04C_refit_000510
[validation_ra

,selected_config_id,source_selected_config_id,candidate_id,model_candidate_id,refit_config_id,metric_equivalence_group_id,target_class,non_target_label,model_family,matrix_family,...,refit_config_duplicate_rank,n_refit_config_candidates,refit_config_candidate_ids,refit_config_candidate_sources,robustness_score,robustness_flags,robustness_rank_in_track,stability_selection_tracks,n_stability_selection_tracks,stability_panel_reason
0,04C_refit_000008,04A_grid_000811,simca_1c2855ffea473c9e,simca_1c2855ffea473c9e,refitcfg_985add5c5b16fa12,NaN,peanut,almond,empirical_cv_rule,object_matrix,...,1,2,"simca_1c2855ffea473c9e,simca_53fda171b13b6751","04A_grid_search,04B_optuna_search",-0.859005,high_fn_rate;high_fp_rate;low_balanced_accuracy,1,object_matrix_2way,1,top_robustness_candidates_per_track
1,04C_refit_000009,optuna_0358,simca_9af4221efa024d1f,simca_9af4221efa024d1f,refitcfg_45a8aa0c5839ab5b,NaN,peanut,almond,rule_variant_grid,object_matrix,...,1,1,simca_9af4221efa024d1f,04B_optuna_search,-0.859005,high_fn_rate;high_fp_rate;low_balanced_accuracy,1,object_matrix_2way,1,top_robustness_candidates_per_track
2,04C_refit_000002,optuna_0284,simca_a207c12cce354da3,simca_a207c12cce354da3,refitcfg_941887f821ed3beb,NaN,peanut,almond,rule_variant_grid,object_matrix,...,1,1,simca_a207c12cce354da3,04B_optuna_search,-0.880274,high_fp_rate;low_balanced_accuracy,2,object_matrix_2way,1,top_robustness_candidates_per_track
3,04C_refit_000003,optuna_0340,simca_08a074054db71001,simca_08a074054db71001,refitcfg_660f259518000307,NaN,peanut,almond,rule_variant_grid,object_matrix,...,1,1,simca_08a074054db71001,04B_optuna_search,-0.880274,high_fp_rate;low_balanced_accuracy,2,object_matrix_2way,1,top_robustness_candidates_per_track
4,04C_refit_000000,optuna_0374,simca_0d9ef35630459e28,simca_0d9ef35630459e28,refitcfg_05835c8f28287c66,metric_eq_001331,peanut,almond,rule_variant_grid,object_matrix,...,1,1,simca_0d9ef35630459e28,04B_optuna_search,-0.890909,high_fp_rate;low_balanced_accuracy,3,object_matrix_2way,1,top_robustness_candidates_per_track
5,04C_refit_000001,optuna_0377,simca_f21f2ff62531f442,simca_f21f2ff62531f442,refitcfg_11ccb5c67c30a9d9,NaN,peanut,almond,rule_variant_grid,object_matrix,...,1,1,simca_f21f2ff62531f442,04B_optuna_search,-0.890909,high_fp_rate;low_balanced_accuracy,3,object_matrix_2way,1,top_robustness_candidates_per_track
6,04C_refit_000004,optuna_0061,simca_138c3b542a51f39b,simca_138c3b542a51f39b,refitcfg_51a2f6db9d7da42e,metric_eq_001562,peanut,almond,rule_variant_grid,object_matrix,...,1,1,simca_138c3b542a51f39b,04B_optuna_search,-0.924185,high_fp_rate;low_balanced_accuracy,4,object_matrix_2way,1,top_robustness_candidates_per_track
7,04C_refit_000010,optuna_0088,simca_b2b3d8faa0b441bd,simca_b2b3d8faa0b441bd,refitcfg_616159cf8e00e05f,metric_eq_001563,peanut,almond,rule_variant_grid,object_matrix,...,1,1,simca_b2b3d8faa0b441bd,04B_optuna_search,-1.077187,high_fn_rate;high_fp_rate;low_balanced_accuracy,5,object_matrix_2way,1,top_robustness_candidates_per_track
8,04C_refit_000005,optuna_0303,simca_36abaf10b8594881,simca_36abaf10b8594881,refitcfg_af87349b4fb80852,NaN,peanut,almond,rule_variant_grid,object_matrix,...,1,1,simca_36abaf10b8594881,04B_optuna_search,-1.087821,high_fp_rate;low_balanced_accuracy,6,object_matrix_2way,1,top_robustness_candidates_per_track
9,04C_refit_000006,04A_grid_000810,simca_c47790494b498010,simca_c47790494b498010,refitcfg_ca5b04b098f4f1af,NaN,peanut,almond,empirical_cv_rule,object_matrix,...,1,1,simca_c47790494b498010,04A_grid_search,-1.087821,high_fp_rate;low_balanced_accuracy,6,object_matrix_2way,1,top_robustness_candidates_per_track


## Optional border/core diagnostics

This block requires detailed 04C batch pixel tables. If they were not saved, the notebook records a skipped status instead of silently pretending the diagnostic was run.


In [9]:
border_core_diagnostics_df = pd.DataFrame()
border_core_status_df = pd.DataFrame()

pixel_batch_paths = (
    sorted(VALIDATION_BATCH_PIXELS_DIR.glob("*.parquet"))
    if VALIDATION_BATCH_PIXELS_DIR.exists()
    else []
)

if RUN_BORDER_CORE_DIAGNOSTICS and pixel_batch_paths:
    border_core_config_ids = (
        candidate_panel_df["selected_config_id"]
        .dropna()
        .astype(str)
        .drop_duplicates()
        .head(MAX_BORDER_CORE_CONFIGS)
        .tolist()
    )

    remaining_config_ids = set(border_core_config_ids)
    object_db, _, _, _ = (
        load_database_for_optional_refits()
    )
    diagnostic_parts = []
    status_parts = []

    for pixel_path in pixel_batch_paths:
        if not remaining_config_ids:
            break

        batch_pixels_df = pd.read_parquet(
            pixel_path,
            filters=[
                (
                    "selected_config_id",
                    "in",
                    sorted(remaining_config_ids),
                )
            ],
        )

        if batch_pixels_df.empty:
            del batch_pixels_df
            continue

        batch_config_ids = set(
            batch_pixels_df["selected_config_id"]
            .dropna()
            .astype(str)
            .unique()
        )

        batch_diagnostics_df, batch_status_df = (
            build_border_core_diagnostics(
                pixel_df=batch_pixels_df,
                object_db=object_db,
                max_configs=None,
            )
        )

        if not batch_diagnostics_df.empty:
            batch_diagnostics_df = batch_diagnostics_df.copy()
            batch_diagnostics_df["pixel_batch_file"] = pixel_path.name
            diagnostic_parts.append(batch_diagnostics_df)

        if not batch_status_df.empty:
            batch_status_df = batch_status_df.copy()
            batch_status_df["pixel_batch_file"] = pixel_path.name
            status_parts.append(batch_status_df)

        remaining_config_ids.difference_update(batch_config_ids)

        del batch_pixels_df
        del batch_diagnostics_df, batch_status_df
        gc.collect()

    border_core_diagnostics_df = concat_nonempty(diagnostic_parts)
    border_core_status_df = concat_nonempty(status_parts)

    if remaining_config_ids:
        missing_status_df = pd.DataFrame(
            {
                "selected_config_id": sorted(remaining_config_ids),
                "border_core_status": "missing_pixels",
                "error": "Configuration not found in batch pixel tables.",
            }
        )
        border_core_status_df = concat_nonempty(
            [border_core_status_df, missing_status_df]
        )

    if border_core_status_df.empty:
        border_core_status_df = pd.DataFrame(
            columns=[
                "selected_config_id",
                "border_core_status",
                "error",
            ]
        )

    del object_db
    gc.collect()

elif RUN_BORDER_CORE_DIAGNOSTICS:
    border_core_status_df = build_border_core_skip_table(
        (
            "04C batch pixel tables are absent. "
            "Rerun 04C with SAVE_BATCH_PIXEL_TABLES=True."
        ),
        pixel_batch_dir=VALIDATION_BATCH_PIXELS_DIR,
    )

else:
    border_core_status_df = build_border_core_skip_table(
        "RUN_BORDER_CORE_DIAGNOSTICS is False.",
        pixel_batch_dir=VALIDATION_BATCH_PIXELS_DIR,
    )

print("Pixel batch files:", len(pixel_batch_paths))
print("Requested configurations:", MAX_BORDER_CORE_CONFIGS)
print("Border/core diagnostics:", border_core_diagnostics_df.shape)
print("Border/core status:", border_core_status_df.shape)

display(border_core_status_df.head())
display(border_core_diagnostics_df.head())

Pixel batch files: 40
Requested configurations: 20
Border/core diagnostics: (100, 27)
Border/core status: (0, 3)


,selected_config_id,border_core_status,error


,selected_config_id,candidate_id,matrix_family,matrix_method,preprocessing,rule_variant,n_components,border_width,zone,object_threshold,...,tn,target_sensitivity,non_target_specificity,balanced_accuracy,accuracy,precision,f1_score,fn_rate,fp_rate,pixel_batch_file
0,04C_refit_000000,simca_0d9ef35630459e28,object_matrix,object_median,absorbance_sg_d2,data_driven_emp_cv,3,0,all_pixels,0.8,...,2,1.0,0.036364,0.518182,0.509259,0.500000,0.666667,0.0,0.963636,batch_0001_pixels.parquet
1,04C_refit_000000,simca_0d9ef35630459e28,object_matrix,object_median,absorbance_sg_d2,data_driven_emp_cv,3,3,core_without_border,0.8,...,2,1.0,0.036364,0.518182,0.509259,0.500000,0.666667,0.0,0.963636,batch_0001_pixels.parquet
2,04C_refit_000000,simca_0d9ef35630459e28,object_matrix,object_median,absorbance_sg_d2,data_driven_emp_cv,3,4,core_without_border,0.8,...,2,1.0,0.036364,0.518182,0.509259,0.500000,0.666667,0.0,0.963636,batch_0001_pixels.parquet
3,04C_refit_000000,simca_0d9ef35630459e28,object_matrix,object_median,absorbance_sg_d2,data_driven_emp_cv,3,2,core_without_border,0.8,...,1,1.0,0.018182,0.509091,0.500000,0.495327,0.662500,0.0,0.981818,batch_0001_pixels.parquet
4,04C_refit_000000,simca_0d9ef35630459e28,object_matrix,object_median,absorbance_sg_d2,data_driven_emp_cv,3,1,core_without_border,0.8,...,0,1.0,0.000000,0.500000,0.490741,0.490741,0.658385,0.0,1.000000,batch_0001_pixels.parquet


## Optional duplicate-candidate review

This analysis reviews the metric-equivalent groups identified in notebook 04C. If the optional duplicated refit check was run in 04C, this table records whether equivalence was verified after refit.


In [10]:
RUN_DUPLICATED_CANDIDATE_ANALYSIS = False

In [11]:
if RUN_DUPLICATED_CANDIDATE_ANALYSIS:
    duplicated_candidate_review_df = build_duplicated_candidate_review(
        groups_df=metric_equivalence_groups_df,
        dropped_df=metric_equivalence_dropped_df,
        refit_comparison_df=duplicated_refit_metric_comparison_df,
    )
    duplicated_candidate_summary_df = summarize_duplicated_candidate_review(
        duplicated_candidate_review_df
    )
else:
    duplicated_candidate_review_df = pd.DataFrame()
    duplicated_candidate_summary_df = pd.DataFrame()

print("duplicated review:", duplicated_candidate_review_df.shape)
print("duplicated summary:", duplicated_candidate_summary_df.shape)
display(duplicated_candidate_summary_df)


duplicated review: (0, 0)
duplicated summary: (0, 0)


""


## Track scoring and review flags

The final table of this notebook combines validation metrics, Pareto status, optional random-state stability, and optional duplicate diagnostics. It is a review table, not a final model selection table.


In [12]:
track_scoring_flags_df = build_track_scoring_table(
    metrics_df=robustness_primary_metrics_df,
    pareto_2way_df=pareto_2way_annotated_df,
    pareto_3way_df=pareto_3way_annotated_df,
    stability_summary_df=random_state_stability_summary_df,
    duplicated_review_df=duplicated_candidate_review_df,
)

print("track scoring flags:", track_scoring_flags_df.shape)
display(
    track_scoring_flags_df[
        [
            col for col in [
                "selection_track",
                "selected_config_id",
                "candidate_id",
                "matrix_family",
                "decision_mode",
                "metric_level",
                "robustness_score",
                "review_rank_in_track",
                "review_flags",
            ]
            if col in track_scoring_flags_df.columns
        ]
    ].head(40)
)


track scoring flags: (3964, 89)


,selection_track,selected_config_id,candidate_id,matrix_family,decision_mode,metric_level,robustness_score,review_rank_in_track,review_flags
0,object_matrix_2way,04C_refit_000008,simca_1c2855ffea473c9e,object_matrix,2way,object,-0.859005,1,high_fn_rate;high_fp_rate;low_balanced_accuracy
1,object_matrix_2way,04C_refit_000009,simca_9af4221efa024d1f,object_matrix,2way,object,-0.859005,1,high_fn_rate;high_fp_rate;low_balanced_accuracy
2,object_matrix_2way,04C_refit_000002,simca_a207c12cce354da3,object_matrix,2way,object,-0.880274,2,high_fp_rate;low_balanced_accuracy
3,object_matrix_2way,04C_refit_000003,simca_08a074054db71001,object_matrix,2way,object,-0.880274,2,high_fp_rate;low_balanced_accuracy
4,object_matrix_2way,04C_refit_000000,simca_0d9ef35630459e28,object_matrix,2way,object,-0.890909,3,high_fp_rate;low_balanced_accuracy
5,object_matrix_2way,04C_refit_000001,simca_f21f2ff62531f442,object_matrix,2way,object,-0.890909,3,high_fp_rate;low_balanced_accuracy
6,object_matrix_2way,04C_refit_000004,simca_138c3b542a51f39b,object_matrix,2way,object,-0.924185,4,high_fp_rate;low_balanced_accuracy
7,object_matrix_2way,04C_refit_000010,simca_b2b3d8faa0b441bd,object_matrix,2way,object,-1.077187,5,dominated_in_pareto;high_fn_rate;high_fp_rate;...
8,object_matrix_2way,04C_refit_000005,simca_36abaf10b8594881,object_matrix,2way,object,-1.087821,6,dominated_in_pareto;high_fp_rate;low_balanced_...
9,object_matrix_2way,04C_refit_000006,simca_c47790494b498010,object_matrix,2way,object,-1.087821,6,dominated_in_pareto;high_fp_rate;low_balanced_...


## Save outputs

The protocol table records the exact switches used for this robustness pass.


In [13]:
robustness_protocol_df = pd.DataFrame([
    {
        "results_tag": RESULTS_TAG,
        "wavelength_mode": WAVELENGTH_MODE,
        "input_04c_dir": str(RESULTS_04C_DIR),
        "run_random_state_stability": bool(RUN_RANDOM_STATE_STABILITY),
        "stability_random_states_json": json.dumps([int(seed) for seed in STABILITY_RANDOM_STATES]),
        "max_stability_candidates_per_track": int(MAX_STABILITY_CANDIDATES_PER_TRACK),
        "run_border_core_diagnostics": bool(RUN_BORDER_CORE_DIAGNOSTICS),
        "max_border_core_configs": int(MAX_BORDER_CORE_CONFIGS),
        "run_duplicated_candidate_analysis": bool(RUN_DUPLICATED_CANDIDATE_ANALYSIS),
        "n_candidate_panel": int(len(candidate_panel_df)),
        "n_validation_metrics": int(len(validation_metrics_df)),
        "n_primary_metrics": int(len(robustness_primary_metrics_df)),
        "n_pareto_2way": int(len(pareto_2way_front_df)),
        "n_pareto_3way": int(len(pareto_3way_front_df)),
        "n_ablation_rows": int(len(ablation_diagnostics_df)),
        "n_stability_panel": int(len(random_state_stability_panel_df)),
        "n_stability_metric_rows": int(len(random_state_stability_metrics_df)),
        "n_border_core_rows": int(len(border_core_diagnostics_df)),
        "n_duplicate_review_rows": int(len(duplicated_candidate_review_df)),
        "n_track_scoring_rows": int(len(track_scoring_flags_df)),
        "pure_test_used": False,
        "final_selection_performed": False,
    }
])

saved_paths = [
    save_parquet(robustness_scored_metrics_df, ROBUSTNESS_SCORED_METRICS_PATH),
    save_parquet(robustness_primary_metrics_df, ROBUSTNESS_PRIMARY_METRICS_PATH),
    save_parquet(pareto_2way_front_df, PARETO_2WAY_FRONT_PATH),
    save_parquet(pareto_2way_annotated_df, PARETO_2WAY_ANNOTATED_PATH),
    save_parquet(pareto_2way_audit_df, PARETO_2WAY_AUDIT_PATH),
    save_parquet(pareto_3way_front_df, PARETO_3WAY_FRONT_PATH),
    save_parquet(pareto_3way_annotated_df, PARETO_3WAY_ANNOTATED_PATH),
    save_parquet(pareto_3way_audit_df, PARETO_3WAY_AUDIT_PATH),
    save_parquet(ablation_diagnostics_df, ABLATION_DIAGNOSTICS_PATH),
    save_parquet(random_state_stability_panel_df, RANDOM_STATE_STABILITY_PANEL_PATH),
    save_parquet(random_state_stability_metrics_df, RANDOM_STATE_STABILITY_METRICS_PATH),
    save_parquet(random_state_stability_summary_df, RANDOM_STATE_STABILITY_SUMMARY_PATH),
    save_parquet(random_state_stability_errors_df, RANDOM_STATE_STABILITY_ERRORS_PATH),
    save_parquet(border_core_diagnostics_df, BORDER_CORE_DIAGNOSTICS_PATH),
    save_parquet(border_core_status_df, BORDER_CORE_STATUS_PATH),
    save_parquet(duplicated_candidate_review_df, DUPLICATED_CANDIDATE_REVIEW_PATH),
    save_parquet(duplicated_candidate_summary_df, DUPLICATED_CANDIDATE_SUMMARY_PATH),
    save_parquet(track_scoring_flags_df, TRACK_SCORING_FLAGS_PATH),
    save_parquet(robustness_protocol_df, ROBUSTNESS_PROTOCOL_PATH),
]

output_inventory_df = pd.DataFrame(
    {
        "path": [str(path) for path in saved_paths],
        "file_name": [Path(path).name for path in saved_paths],
    }
)
display(robustness_protocol_df)
display(output_inventory_df)


,results_tag,wavelength_mode,input_04c_dir,run_random_state_stability,stability_random_states_json,max_stability_candidates_per_track,run_border_core_diagnostics,max_border_core_configs,run_duplicated_candidate_analysis,n_candidate_panel,...,n_pareto_2way,n_pareto_3way,n_ablation_rows,n_stability_panel,n_stability_metric_rows,n_border_core_rows,n_duplicate_review_rows,n_track_scoring_rows,pure_test_used,final_selection_performed
0,non_noisy_all,non_noisy_all,C:\Users\alixg\OneDrive - Université Paris-Dau...,True,"[0, 1, 2, 3, 4, 5, 10, 20, 42, 100]",12,True,20,False,1982,...,61,359,131,38,380,100,0,3964,False,False


,path,file_name
0,C:\Users\alixg\OneDrive - Université Paris-Dau...,robustness_scored_metrics.parquet
1,C:\Users\alixg\OneDrive - Université Paris-Dau...,robustness_primary_metrics.parquet
2,C:\Users\alixg\OneDrive - Université Paris-Dau...,pareto_2way_front.parquet
3,C:\Users\alixg\OneDrive - Université Paris-Dau...,pareto_2way_annotated.parquet
4,C:\Users\alixg\OneDrive - Université Paris-Dau...,pareto_2way_audit.parquet
5,C:\Users\alixg\OneDrive - Université Paris-Dau...,pareto_3way_front.parquet
6,C:\Users\alixg\OneDrive - Université Paris-Dau...,pareto_3way_annotated.parquet
7,C:\Users\alixg\OneDrive - Université Paris-Dau...,pareto_3way_audit.parquet
8,C:\Users\alixg\OneDrive - Université Paris-Dau...,ablation_diagnostics.parquet
9,C:\Users\alixg\OneDrive - Université Paris-Dau...,random_state_stability_panel.parquet
